# Notebook 22: Natural Language Processing Fundamentals (TF-IDF → Embeddings)
### Part 22/30 – ML Mastery Series for Python Experts

## From Words to Numbers – The Evolution of Text Representation

You taught models to see images and remember sequences — now teach them to read and understand human language, from crude word counts to rich semantic vectors…

- **Text is unstructured** → need numerical features: Computers don't understand words; we must convert text to tensors while preserving meaning
- **Bag-of-words loses order & semantics**: "Dog bites man" = "Man bites dog" in BoW, yet opposite meanings
- **TF-IDF improves weighting**: Down-weights common words (the, and), up-weights discriminative terms (rare but informative words)
- **Embeddings capture similarity & analogy**: Dense vectors where cosine distance ≈ semantic similarity; king - man + woman ≈ queen
- **Contextual vs static embeddings**: Static (GloVe, Word2Vec) give one vector per word; contextual (BERT) give different vectors per usage
- **Transition from sparse high-dim to dense low-dim**: 10k-dimensional sparse TF-IDF → 100-300 dimensional dense embeddings
- **Huge performance jump on downstream tasks**: Embeddings enable transfer learning in NLP, just like ImageNet did for vision
- **Distributional hypothesis**: "You shall know a word by the company it keeps" — words appearing in similar contexts have similar meanings

## Learning Objectives

By the end of this notebook, you will be able to:

- **Preprocess text** (tokenize, lowercase, remove stopwords/punctuation) for clean feature extraction
- **Build TF-IDF vectors with sklearn** and understand the sparse matrix representation
- **Train classical ML models** (Logistic Regression, SVM) on TF-IDF features as strong baselines
- **Understand word embedding intuition**: Distributional hypothesis and why dense vectors capture semantics
- **Load & use pre-trained GloVe/FastText** embeddings via gensim for zero-shot transfer
- **Train trainable embeddings in Keras** end-to-end with backpropagation for your specific task
- **Compare TF-IDF vs embeddings** on classification accuracy and training dynamics
- **Visualize embeddings with PCA/t-SNE/UMAP** to discover semantic clusters (countries, professions, emotions)
- **Compute word similarity & analogy** with cosine similarity and vector arithmetic
- **Know limitations of static embeddings**: Polysemy, out-of-vocabulary words, lack of context-sensitivity

## 📝 1. Text Preprocessing & TF-IDF Basics

TF-IDF (Term Frequency-Inverse Document Frequency) is the workhorse of classical NLP. It converts documents into weighted word-count vectors, where rare but discriminative words score higher than common stopwords.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import gensim.downloader as api
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import re
import string
import warnings
warnings.filterwarnings('ignore')

# Set seeds and style
tf.random.set_seed(42)
np.random.seed(42)
sns.set_theme(style="whitegrid")
%matplotlib inline

print(f"TensorFlow version: {tf.__version__}")
print(f"Gensim available for embeddings: True")

In [ ]:
# Load IMDB sentiment dataset (we'll use the text version via keras)
# keras.datasets.imdb gives sequences; we'll reconstruct text for TF-IDF demo
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {v: k for k, v in word_index.items()}

# Load IMDB data
(x_train_seq, y_train), (x_test_seq, y_test) = keras.datasets.imdb.load_data(num_words=10000)

# Decode sequences back to text for sklearn processing
def decode_review(encoded_seq):
    """Convert integer sequence back to text string."""
    words = []
    for i in encoded_seq:
        if i == 1:  # START token
            continue
        word = reverse_word_index.get(i - 3, '?')  # offset by 3 for special tokens
        words.append(word)
    return ' '.join(words)

# Decode a subset for faster TF-IDF processing (full dataset is 25k samples)
subset_size = 5000
x_train_text = [decode_review(seq) for seq in x_train_seq[:subset_size]]
x_test_text = [decode_review(seq) for seq in x_test_seq[:subset_size//5]]
y_train_sub = y_train[:subset_size]
y_test_sub = y_test[:subset_size//5]

print(f"Sample review: {x_train_text[0][:200]}...")
print(f"Label: {y_train_sub[0]} (0=negative, 1=positive)")
print(f"Training samples: {len(x_train_text)}, Test samples: {len(x_test_text)}")

In [ ]:
# Build TF-IDF vectors with sklearn
# TfidfVectorizer combines CountVectorizer + TfidfTransformer
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,      # Keep top 10k words by frequency
    stop_words='english',  # Remove common English stopwords
    lowercase=True,        # Normalize case
    ngram_range=(1, 1),    # Use unigrams only (single words)
    min_df=2,              # Ignore words appearing in <2 documents
    max_df=0.95            # Ignore words appearing in >95% of docs (too common)
)

# Fit on training data, transform both train and test
X_train_tfidf = tfidf_vectorizer.fit_transform(x_train_text)
X_test_tfidf = tfidf_vectorizer.transform(x_test_text)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"Matrix type: {type(X_train_tfidf)} (sparse CSR format)")
print(f"Non-zero entries: {X_train_tfidf.nnz:,} ({100*X_train_tfidf.nnz/(X_train_tfidf.shape[0]*X_train_tfidf.shape[1]):.2f}% dense)")

# Show top TF-IDF features for a sample document
feature_names = tfidf_vectorizer.get_feature_names_out()
sample_vector = X_train_tfidf[0].toarray().flatten()
top_indices = sample_vector.argsort()[-10:][::-1]
print(f"\nTop TF-IDF features in sample review:")
for idx in top_indices:
    if sample_vector[idx] > 0:
        print(f"  {feature_names[idx]}: {sample_vector[idx]:.4f}")

In [ ]:
# Train classical ML models on TF-IDF features
# Model 1: Logistic Regression (linear classifier, fast and strong baseline)
lr_model = LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1)
lr_model.fit(X_train_tfidf, y_train_sub)

# Model 2: Multinomial Naive Bayes (probabilistic, works well with word counts)
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train_sub)

# Evaluate both models
lr_pred = lr_model.predict(X_test_tfidf)
nb_pred = nb_model.predict(X_test_tfidf)

lr_acc = accuracy_score(y_test_sub, lr_pred)
nb_acc = accuracy_score(y_test_sub, nb_pred)

print(f"Logistic Regression accuracy: {lr_acc:.4f}")
print(f"Naive Bayes accuracy: {nb_acc:.4f}")
print(f"\nLogistic Regression Classification Report:")
print(classification_report(y_test_sub, lr_pred, target_names=['Negative', 'Positive']))

## 🎯 2. TF-IDF + Classical ML – Strong Baseline

Classical ML on TF-IDF often beats deep learning on small datasets. Let's optimize our pipeline with n-grams and hyperparameter search, then interpret what features the model learns.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# Advanced TF-IDF with bigrams (n-gram range 1-2 captures phrases like "not good")
advanced_tfidf = TfidfVectorizer(
    max_features=15000,
    stop_words='english',
    ngram_range=(1, 2),    # Include unigrams AND bigrams
    min_df=3,
    max_df=0.9,
    sublinear_tf=True      # Use sublinear tf scaling (1 + log(tf))
)

# Build pipeline for easy grid search
pipeline = Pipeline([
    ('tfidf', advanced_tfidf),
    ('clf', LinearSVC())
)

# Grid search on smaller subset for speed
param_grid = {
    'tfidf__max_features': [5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.1, 1.0, 10.0]
}

print("Running GridSearchCV on TF-IDF + LinearSVC...")
grid_search = GridSearchCV(
    pipeline, param_grid,
    cv=3, scoring='accuracy',
    n_jobs=-1, verbose=0
)

# Use smaller subset for grid search
grid_search.fit(x_train_text[:2000], y_train_sub[:2000])

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

# Evaluate best model on full test set
best_model = grid_search.best_estimator_
best_pred = best_model.predict(x_test_text)
best_acc = accuracy_score(y_test_sub, best_pred)
print(f"Test accuracy with best params: {best_acc:.4f}")

In [ ]:
# Interpret model: Most important features for each class
# Extract coefficients from the LinearSVC in the pipeline
svc_model = best_model.named_steps['clf']
tfidf_model = best_model.named_steps['tfidf']
feature_names = tfidf_model.get_feature_names_out()

# Get top positive and negative coefficients
coef = svc_model.coef_.flatten()
top_positive = coef.argsort()[-15:][::-1]
top_negative = coef.argsort()[:15]

print("Top features indicating POSITIVE sentiment:")
for idx in top_positive:
    print(f"  {feature_names[idx]}: {coef[idx]:.4f}")

print("\nTop features indicating NEGATIVE sentiment:")
for idx in top_negative:
    print(f"  {feature_names[idx]}: {coef[idx]:.4f}")

# Visualize confusion matrix
cm = confusion_matrix(y_test_sub, best_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix: TF-IDF + LinearSVC')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 🧬 3. Introducing Word Embeddings – Static Pre-trained

TF-IDF gives sparse, high-dimensional vectors with no notion of word similarity. Embeddings are dense, low-dimensional (50-300d), and semantically meaningful: similar words cluster together. We'll use GloVe (Global Vectors) pre-trained on Wikipedia + Gigaword.

In [ ]:
# Load pre-trained GloVe embeddings via gensim downloader
# This downloads ~100MB on first run
print("Loading pre-trained GloVe embeddings (100d)...")
print("This may take a minute on first download...")

try:
    # Load lightweight 100d GloVe vectors
    glove_vectors = api.load("glove-wiki-gigaword-100")
    print(f"Loaded GloVe: {len(glove_vectors)} words, {glove_vectors.vector_size} dimensions")
    
    # Test similarity
    print(f"\n'vector' - 'man' + 'woman' ≈ ?")
    result = glove_vectors.most_similar(positive=['vector', 'woman'], negative=['man'], topn=3)
    for word, score in result:
        print(f"  {word}: {score:.4f}")
        
except Exception as e:
    print(f"Gensim download failed: {e}")
    print("Continuing with simulated embeddings for demonstration...")
    glove_vectors = None

In [ ]:
# Build document vectors by averaging word embeddings
def document_to_vector(text, embeddings, dim=100):
    """
    Convert document to vector by averaging word embeddings.
    Words not in embedding vocabulary are skipped (OOV handling).
    """
    words = text.lower().split()
    vectors = []
    
    for word in words:
        if word in embeddings:
            vectors.append(embeddings[word])
    
    if len(vectors) == 0:
        return np.zeros(dim)  # Return zero vector if no words found
    
    return np.mean(vectors, axis=0)

# Convert all documents to GloVe vectors (if loaded)
if glove_vectors is not None:
    print("Converting documents to GloVe vectors...")
    
    # Process in batches for memory efficiency
    X_train_glove = np.array([document_to_vector(text, glove_vectors) for text in x_train_text])
    X_test_glove = np.array([document_to_vector(text, glove_vectors) for text in x_test_text])
    
    print(f"GloVe document vectors shape: {X_train_glove.shape}")
    
    # Train classifier on averaged GloVe vectors
    from sklearn.linear_model import LogisticRegression
    glove_lr = LogisticRegression(max_iter=1000, C=1.0)
    glove_lr.fit(X_train_glove, y_train_sub)
    
    glove_pred = glove_lr.predict(X_test_glove)
    glove_acc = accuracy_score(y_test_sub, glove_pred)
    
    print(f"\nGloVe (averaged) + Logistic Regression accuracy: {glove_acc:.4f}")
    print(f"TF-IDF + Logistic Regression accuracy: {lr_acc:.4f}")
    print(f"GloVe captures semantics but loses word order (bag-of-embeddings)")
else:
    print("Skipping GloVe experiment (embeddings not available)")
    glove_acc = None

In [ ]:
# Demonstrate word similarity and analogy with GloVe
if glove_vectors is not None:
    print("="*60)
    print("WORD SIMILARITY & ANALOGY DEMONSTRATIONS")
    print("="*60)
    
    # Similarity
    print("\n1. Most similar to 'excellent':")
    for word, score in glove_vectors.most_similar('excellent', topn=5):
        print(f"   {word}: {score:.4f}")
    
    print("\n2. Most similar to 'terrible':")
    for word, score in glove_vectors.most_similar('terrible', topn=5):
        print(f"   {word}: {score:.4f}")
    
    # Analogy: king - man + woman = queen
    print("\n3. Analogy: king - man + woman ≈ ?")
    analogy_result = glove_vectors.most_similar(
        positive=['king', 'woman'], 
        negative=['man'], 
        topn=3
    )
    for word, score in analogy_result:
        print(f"   {word}: {score:.4f}")
    
    # Sentiment-specific analogy
    print("\n4. Analogy: good - bad + acting ≈ ?")
    analogy_result = glove_vectors.most_similar(
        positive=['good', 'acting'], 
        negative=['bad'], 
        topn=3
    )
    for word, score in analogy_result:
        print(f"   {word}: {score:.4f}")
        
    # Cosine similarity formula demonstration
    print(f"\n5. Cosine similarity between 'happy' and 'joyful':")
    sim = glove_vectors.similarity('happy', 'joyful')
    print(f"   similarity = {sim:.4f}")
    print(f"   Formula: cos(θ) = (A·B) / (||A|| ||B||)")

## 🔧 4. Trainable Embeddings in Keras – End-to-End

Instead of pre-trained embeddings, let's learn embeddings specifically for our sentiment task. The Embedding layer maps integer word indices to dense vectors, learned via backpropagation. This is end-to-end deep learning for NLP.

In [ ]:
# Prepare data for Keras embedding layer
# Tokenizer converts text to sequences of integers
vocab_size = 10000
maxlen = 200  # Pad/truncate to 200 words

# Use Keras Tokenizer on our text
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(x_train_text)

# Convert text to sequences
X_train_seq = tokenizer.texts_to_sequences(x_train_text)
X_test_seq = tokenizer.texts_to_sequences(x_test_text)

# Pad sequences to fixed length
X_train_pad = pad_sequences(X_train_seq, maxlen=maxlen, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=maxlen, padding='post', truncating='post')

print(f"Padded sequences shape: {X_train_pad.shape}")
print(f"Example sequence (first 10 tokens): {X_train_pad[0][:10]}")
print(f"Sequence lengths: min={np.min(np.sum(X_train_pad > 0, axis=1))}, "
      f"max={np.max(np.sum(X_train_pad > 0, axis=1))}, "
      f"mean={np.mean(np.sum(X_train_pad > 0, axis=1)):.1f}")

In [ ]:
# Build model with trainable embedding layer
def build_embedding_model(vocab_size, embedding_dim=128, maxlen=200):
    """
    Simple embedding + pooling model for sentiment classification.
    Embedding layer learns 128-dim vectors for each word during training.
    """
    model = keras.Sequential([
        # Embedding layer: input_dim=vocab_size, output_dim=embedding_dim
        # Each word index gets mapped to a dense 128-dimensional vector
        layers.Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            input_length=maxlen,
            name='embedding'
        ),
        
        # GlobalAveragePooling1D averages word embeddings across sequence
        # This is 'bag-of-embeddings' approach (loses order but efficient)
        layers.GlobalAveragePooling1D(),
        
        # Dense layers for classification
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

model_embed = build_embedding_model(vocab_size)
model_embed.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_embed.summary()

In [ ]:
# Train the embedding model
print("Training trainable embedding model...")
history_embed = model_embed.fit(
    X_train_pad, y_train_sub,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    ],
    verbose=0
)

# Evaluate
embed_loss, embed_acc = model_embed.evaluate(X_test_pad, y_test_sub, verbose=0)
print(f"Trainable Embedding + GlobalAveragePooling accuracy: {embed_acc:.4f}")
print(f"TF-IDF + Logistic Regression accuracy: {lr_acc:.4f}")
if glove_acc:
    print(f"GloVe (averaged) accuracy: {glove_acc:.4f}")

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_embed.history['accuracy'], label='Train')
axes[0].plot(history_embed.history['val_accuracy'], label='Val')
axes[0].set_title('Embedding Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_embed.history['loss'], label='Train')
axes[1].plot(history_embed.history['val_loss'], label='Val')
axes[1].set_title('Embedding Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🎨 5. Visualizing & Interpreting Embeddings

Let's extract the learned embedding weights and visualize them in 2D using PCA. Words with similar sentiment or topic should cluster together.

In [ ]:
# Extract learned embedding weights
embedding_layer = model_embed.get_layer('embedding')
embedding_weights = embedding_layer.get_weights()[0]  # Shape: (vocab_size, embedding_dim)

print(f"Embedding matrix shape: {embedding_weights.shape}")

# Create word-to-index mapping from tokenizer
word_index_map = tokenizer.word_index
index_word_map = {v: k for k, v in word_index_map.items()}

# Select words to visualize (frequent words with clear sentiment)
words_to_plot = [
    # Positive sentiment words
    'excellent', 'amazing', 'wonderful', 'great', 'best', 'love', 'perfect', 'beautiful',
    # Negative sentiment words  
    'terrible', 'awful', 'worst', 'bad', 'hate', 'boring', 'poor', 'waste',
    # Neutral/objective words
    'movie', 'film', 'story', 'acting', 'plot', 'character', 'scene', 'director',
    # Intensifiers
    'very', 'really', 'quite', 'extremely', 'totally'
]

# Filter to words actually in our vocabulary
words_in_vocab = [w for w in words_to_plot if w in word_index_map and word_index_map[w] < vocab_size]
word_indices = [word_index_map[w] for w in words_in_vocab]
word_vectors = embedding_weights[word_indices]

print(f"Plotting {len(words_in_vocab)} words from vocabulary")

# Reduce to 2D with PCA
pca = PCA(n_components=2)
word_vectors_2d = pca.fit_transform(word_vectors)

# Plot
plt.figure(figsize=(12, 10))
colors = ['green' if w in words_to_plot[:8] else 'red' if w in words_to_plot[8:16] else 'blue' 
          for w in words_in_vocab]

plt.scatter(word_vectors_2d[:, 0], word_vectors_2d[:, 1], c=colors, alpha=0.6, s=100)

# Annotate points
for i, word in enumerate(words_in_vocab):
    plt.annotate(word, (word_vectors_2d[i, 0], word_vectors_2d[i, 1]), 
                fontsize=10, ha='center', va='bottom')

plt.title('Learned Embeddings Visualized with PCA\n(Green=Positive, Red=Negative, Blue=Neutral)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"PCA explains {sum(pca.explained_variance_ratio_)*100:.1f}% of variance in 2D")

In [ ]:
# Compare learned embeddings with GloVe using t-SNE (if available)
if glove_vectors is not None:
    # Get GloVe vectors for same words
    glove_subset = []
    glove_words = []
    for w in words_in_vocab:
        if w in glove_vectors:
            glove_subset.append(glove_vectors[w])
            glove_words.append(w)
    
    if len(glove_subset) > 10:
        glove_subset = np.array(glove_subset)
        
        # t-SNE for better visualization (non-linear dimensionality reduction)
        from sklearn.manifold import TSNE
        
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(10, len(words_in_vocab)-1))
        
        # Fit on combined data to ensure same projection
        combined = np.vstack([word_vectors[:len(glove_subset)], glove_subset])
        combined_2d = tsne.fit_transform(combined)
        
        learned_2d = combined_2d[:len(glove_subset)]
        glove_2d = combined_2d[len(glove_subset):]
        
        # Plot side by side
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
        
        # Learned embeddings
        ax1.scatter(learned_2d[:, 0], learned_2d[:, 1], c='blue', alpha=0.6, s=100)
        for i, word in enumerate(glove_words):
            ax1.annotate(word, (learned_2d[i, 0], learned_2d[i, 1]), fontsize=9)
        ax1.set_title('Task-Specific Learned Embeddings (t-SNE)')
        ax1.grid(True, alpha=0.3)
        
        # GloVe embeddings
        ax2.scatter(glove_2d[:, 0], glove_2d[:, 1], c='green', alpha=0.6, s=100)
        for i, word in enumerate(glove_words):
            ax2.annotate(word, (glove_2d[i, 0], glove_2d[i, 1]), fontsize=9)
        ax2.set_title('Pre-trained GloVe Embeddings (t-SNE)')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print("Left: Embeddings learned specifically for sentiment (task-specific)")
        print("Right: General pre-trained embeddings (transfer learning)")

## 🔄 6. Sequence Modeling with Embeddings – LSTM/GRU

Averaging embeddings loses word order ("not good" = "good not"). Let's add an LSTM on top of embeddings to capture sequence information and long-range dependencies.

In [ ]:
# Build embedding + LSTM model (respects word order)
def build_lstm_embedding_model(vocab_size, embedding_dim=128, maxlen=200):
    """LSTM model that uses embeddings then processes sequence."""
    model = keras.Sequential([
        # Same embedding layer as before
        layers.Embedding(vocab_size, embedding_dim, input_length=maxlen),
        
        # LSTM processes the sequence of embeddings
        # return_sequences=False means we only use the final hidden state
        layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2),
        
        # Classification head
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

model_lstm = build_lstm_embedding_model(vocab_size)
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Training Embedding + LSTM model...")
history_lstm = model_lstm.fit(
    X_train_pad, y_train_sub,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=0
)

lstm_loss, lstm_acc = model_lstm.evaluate(X_test_pad, y_test_sub, verbose=0)
print(f"Embedding + LSTM accuracy: {lstm_acc:.4f}")
print(f"Embedding + AveragePooling accuracy: {embed_acc:.4f}")
print(f"Improvement from sequence modeling: +{lstm_acc - embed_acc:.4f}")

In [ ]:
# Try Bidirectional LSTM for even better performance
def build_bilstm_model(vocab_size, embedding_dim=128, maxlen=200):
    """Bidirectional LSTM processes text in both directions."""
    model = keras.Sequential([
        layers.Embedding(vocab_size, embedding_dim, input_length=maxlen),
        
        # Bidirectional wrapper runs two LSTMs: one forward, one backward
        layers.Bidirectional(layers.LSTM(64, dropout=0.2)),
        
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

model_bilstm = build_bilstm_model(vocab_size)
model_bilstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Training Embedding + Bidirectional LSTM...")
history_bilstm = model_bilstm.fit(
    X_train_pad, y_train_sub,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=0
)

bilstm_loss, bilstm_acc = model_bilstm.evaluate(X_test_pad, y_test_sub, verbose=0)
print(f"Embedding + Bidirectional LSTM accuracy: {bilstm_acc:.4f}")

In [ ]:
# Final comparison of all approaches
results = {
    'Method': [
        'TF-IDF + Logistic Regression',
        'TF-IDF + LinearSVC (tuned)',
        'GloVe (averaged) + LR',
        'Trainable Embedding + AveragePool',
        'Trainable Embedding + LSTM',
        'Trainable Embedding + BiLSTM'
    ],
    'Accuracy': [lr_acc, best_acc, glove_acc if glove_acc else 0, embed_acc, lstm_acc, bilstm_acc],
    'Type': ['Sparse', 'Sparse', 'Dense Pre-trained', 'Dense Learned', 'Sequential', 'Sequential']
}

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Accuracy', ascending=False)

print("\n" + "="*70)
print("FINAL COMPARISON: All NLP Approaches")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

# Visualize comparison
plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if t == 'Sparse' else '#3498db' if 'Pre' in t else '#2ecc71' 
          for t in results_df['Type']]
bars = plt.barh(results_df['Method'], results_df['Accuracy'], color=colors, edgecolor='black')
plt.xlabel('Test Accuracy')
plt.title('NLP Method Comparison on IMDB Sentiment')
plt.xlim(0.5, 1.0)

# Add value labels
for bar, acc in zip(bars, results_df['Accuracy']):
    plt.text(acc + 0.01, bar.get_y() + bar.get_height()/2, 
             f'{acc:.3f}', va='center', fontweight='bold')

plt.legend(['Sparse (TF-IDF)', 'Dense Pre-trained', 'Dense Learned/Sequential'], 
           loc='lower right')
plt.tight_layout()
plt.show()

## 🔍 7. Similarity & Retrieval – Cosine on Embeddings

Embeddings enable semantic search: find documents similar to a query by comparing vector representations. Cosine similarity measures the angle between vectors: $cos(θ) = \frac{A \cdot B}{||A|| ||B||}$

In [ ]:
# Compute cosine similarity between documents using learned embeddings
from sklearn.metrics.pairwise import cosine_similarity

# Create a model that outputs the pooled embedding (before final classification)
embedding_model = keras.Model(
    inputs=model_embed.input,
    outputs=model_embed.layers[1].output  # GlobalAveragePooling1D output
)

# Get document embeddings for test set
doc_embeddings = embedding_model.predict(X_test_pad, verbose=0, batch_size=32)
print(f"Document embedding shape: {doc_embeddings.shape}")

# Function to find similar documents
def find_similar_documents(query_idx, doc_embeddings, top_k=5):
    """Find top-k most similar documents to query using cosine similarity."""
    query_vec = doc_embeddings[query_idx:query_idx+1]
    similarities = cosine_similarity(query_vec, doc_embeddings).flatten()
    
    # Get top-k (excluding the query itself)
    top_indices = similarities.argsort()[-top_k-1:][::-1]
    top_indices = [i for i in top_indices if i != query_idx][:top_k]
    
    return top_indices, similarities[top_indices]

# Example: Find similar reviews
query_idx = 0  # First test review
similar_indices, similar_scores = find_similar_documents(query_idx, doc_embeddings, top_k=3)

print("Query review:")
print(f"  {x_test_text[query_idx][:150]}...")
print(f"  Sentiment: {'Positive' if y_test_sub[query_idx] else 'Negative'}")
print(f"\nTop 3 similar reviews:")
for i, (idx, score) in enumerate(zip(similar_indices, similar_scores)):
    print(f"\n{i+1}. Similarity: {score:.4f}")
    print(f"   {x_test_text[idx][:150]}...")
    print(f"   Sentiment: {'Positive' if y_test_sub[idx] else 'Negative'}")

In [ ]:
# Semantic search: Find reviews most similar to a query word/phrase
def semantic_search(query_word, tokenizer, embedding_model, X_data, texts, top_k=5):
    """Find documents most related to a query word using embeddings."""
    # Get embedding for query word (if in vocabulary)
    if query_word in tokenizer.word_index:
        word_idx = tokenizer.word_index[query_word]
        # Create one-hot input for the word
        query_input = np.zeros((1, maxlen))
        query_input[0, 0] = word_idx
        query_vec = embedding_model.predict(query_input, verbose=0)
    else:
        print(f"Word '{query_word}' not in vocabulary")
        return []
    
    # Compare to all documents
    similarities = cosine_similarity(query_vec, doc_embeddings).flatten()
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    return top_indices, similarities[top_indices]

# Search for reviews similar to "excellent"
print("Semantic search for 'excellent':")
indices, scores = semantic_search('excellent', tokenizer, embedding_model, X_test_pad, x_test_text)
for idx, score in zip(indices[:3], scores[:3]):
    sentiment = 'Positive' if y_test_sub[idx] else 'Negative'
    print(f"  Score: {score:.4f} [{sentiment}] {x_test_text[idx][:100]}...")

print("\nSemantic search for 'terrible':")
indices, scores = semantic_search('terrible', tokenizer, embedding_model, X_test_pad, x_test_text)
for idx, score in zip(indices[:3], scores[:3]):
    sentiment = 'Positive' if y_test_sub[idx] else 'Negative'
    print(f"  Score: {score:.4f} [{sentiment}] {x_test_text[idx][:100]}...")

print("\nNote: Semantic search finds conceptually related documents,")
print("not just documents containing the exact query word.")

## ⚠️ Common Pitfalls & Pro Tips

- **Not lowercasing/stemming/lemmatizing** → "The" and "the" become different tokens, splitting your vocabulary and reducing statistical power
- **Too few max_features** → Lose rare but discriminative words. Too many → noise and overfitting. Start with 10k-50k for medium datasets.
- **OOV words in pre-trained embeddings** → Words not in GloVe get zero vectors or random initialization. Use `<UNK>` token or subword embeddings (FastText).
- **Average pooling loses word order** → "not good" and "good not" have same average embedding. Use RNN/CNN/Transformer for order-sensitive tasks.
- **Not padding/masking sequences** → Short sequences padded with zeros can bias gradients if not masked. Use `mask_zero=True` in Embedding layer.
- **GloVe vs FastText for rare words** → GloVe struggles with rare words; FastText uses subword information (character n-grams) to handle OOV better.
- **Embeddings not magic on very small data** → With <1k samples, TF-IDF + LinearSVC often beats deep embeddings due to fewer parameters.
- **Forgetting to freeze pre-trained embeddings** → If using GloVe with small data, set `trainable=False` in Embedding layer to prevent overfitting.
- **Ignoring sequence length distribution** → Padding everything to 500 when 90% of sequences are <50 wastes compute. Analyze length distribution first.
- **Not normalizing vectors before cosine similarity** → Cosine similarity assumes normalized vectors. Use `sklearn.metrics.pairwise.cosine_similarity` which handles this.
- **Using embeddings for out-of-domain data** → Medical embeddings for legal text won't work. Domain-specific embeddings (BioBERT, Legal-BERT) exist for reason.
- **Static vs contextual embeddings confusion** → GloVe gives same vector for "bank" (river) and "bank" (money). Use BERT for context-dependent meanings.

## 📝 Exercises

Practice your NLP skills:

**Easy:** Build TF-IDF + LinearSVC on the full 20 Newsgroups dataset (sklearn.datasets.fetch_20newsgroups). Aim for >80% accuracy. Compare unigrams vs. bigrams performance.

**Medium:** Train an Embedding + GlobalAveragePooling model on IMDB with different embedding dimensions (32, 64, 128, 256). Plot accuracy vs. dimensionality and analyze the trade-off.

**Medium:** Visualize 50 words from the IMDB vocabulary in 2D using t-SNE. Color-code by sentiment (positive/negative/neutral) and label distinct clusters you observe.

**Hard:** Add a Bidirectional LSTM on top of trainable embeddings. Implement early stopping with model checkpointing. Beat the TF-IDF baseline (>87% accuracy) and document your hyperparameter choices.

**Bonus:** Implement a simple word analogy solver using your trained embeddings. Given 'man', 'woman', 'king', predict 'queen' via vector arithmetic (king - man + woman ≈ ?). Test on 5 different analogies.

<details>
<summary><b>Exercise Solutions (Click to Expand)</b></summary>

### Easy Solution Outline (20 Newsgroups)
```python
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

# Load data
newsgroups = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
X, y = newsgroups.data, newsgroups.target

# TF-IDF + LinearSVC
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words='english')
X_tfidf = vectorizer.fit_transform(X)
clf = LinearSVC(C=0.5)
clf.fit(X_tfidf, y)
```

### Hard Solution Outline (BiLSTM with Checkpointing)
```python
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_accuracy')
early_stop = EarlyStopping(patience=5, restore_best_weights=True)

model = keras.Sequential([
    layers.Embedding(10000, 128, input_length=200),
    layers.Bidirectional(layers.LSTM(64, dropout=0.3)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, validation_split=0.2, epochs=20, callbacks=[checkpoint, early_stop])
```

### Bonus Solution Outline (Word Analogy)
```python
def word_analogy(embeddings, word_a, word_b, word_c, topn=1):
    # analogy: a is to b as c is to ?
    # vector = b - a + c
    vec = embeddings[word_b] - embeddings[word_a] + embeddings[word_c]
    return embeddings.similar_by_vector(vec, topn=topn)
    
# Example: word_analogy(glove_vectors, 'man', 'woman', 'king')
# Returns: [('queen', 0.78), ...]
```

</details>

## ✅ Summary – What You Learned Today

- **Text representation evolution**: From sparse TF-IDF vectors to dense semantic embeddings — the foundation of modern NLP
- **TF-IDF as strong baseline**: Classical ML on TF-IDF often beats deep learning on small datasets; n-grams capture local word order
- **Word embedding intuition**: Distributional hypothesis — words in similar contexts have similar meanings, represented as dense vectors
- **Pre-trained embeddings**: GloVe and FastText provide transfer learning for NLP, enabling zero-shot and few-shot learning
- **End-to-end learned embeddings**: Keras Embedding layers trained via backpropagation adapt specifically to your task
- **Sequence modeling matters**: Averaging embeddings (bag-of-embeddings) loses word order; LSTMs/GRUs capture sequential dependencies
- **Bidirectional processing**: Future context often critical for understanding — bidirectional RNNs process text both ways
- **Semantic similarity**: Cosine similarity on embedding vectors enables semantic search, clustering, and analogies
- **Visualization techniques**: PCA and t-SNE reveal semantic clusters in embedding space (sentiment, topic, syntactic categories)
- **Trade-offs**: Sparse vs. dense, static vs. contextual, pre-trained vs. learned — choose based on data size, domain, and task

## 🔮 Next Notebook Preview

**Notebook 23: Time Series Forecasting – ARIMA, Prophet, and Deep Learning**

Moving from text back to temporal data with specialized techniques:
- **Classical methods**: ARIMA, SARIMA, and exponential smoothing for baseline forecasts
- **Facebook Prophet**: Automated forecasting with seasonality and holiday effects
- **Deep learning for time series**: LSTM, GRU, and Temporal Convolutional Networks (TCN)
- **Stationarity and preprocessing**: Differencing, decomposition, and feature engineering
- **Evaluation metrics**: MAE, RMSE, MAPE, and directional accuracy for business impact
- **Multi-step and probabilistic forecasting**: Prediction intervals and uncertainty quantification

Prepare to predict the future with statistical rigor and neural power! 📈🔮